# 01 - Baseline VLM

Charge MedGemma, applique le prompt baseline sur CheXpert et calcule l accuracy.

In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [2]:
import json, re
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM

# Modele charge UNE seule fois (4B parametres, GPU Kaggle requis)
MODEL_ID = "google/medgemma-1.5-4b-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto")
# Si cette classe erreur selon ta version de transformers, remplace par AutoModelForImageTextToText


The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [3]:
PROMPT_BASELINE = """You are an educational radiology assistant for engineering students.
You are not a clinician and you must not provide a definitive diagnosis.
Analyze the provided frontal chest X-ray: normal vs suspected lung opacity/pneumonia-related abnormality vs uncertain.
Return only valid JSON with this schema:
{"image_quality":"good|limited|poor","predicted_class":"normal|suspected_opacity|uncertain","confidence":0.0,"visual_evidence":["obs"],"justification":"2-4 cautious sentences","limitations":["lim"],"warning":"Educational prototype only. Not for diagnosis."}
Rules:
- Do not invent patient history.
- Do not mention findings that are not visible.
- Use "uncertain" when image quality is poor or evidence is weak.
- Keep the response concise."""

PROMPT_IMPROVED = """You are an educational radiology assistant for engineering students.
You are not a clinician and you must not provide a definitive diagnosis.
Analyze the frontal chest X-ray under strict uncertainty rules.
Before deciding, check whether the apparent abnormality could be due to projection, rotation, poor exposure, low inspiration or overlapping anatomy. If so, reduce confidence and consider "uncertain".
Return only valid JSON with this schema:
{"image_quality":"good|limited|poor","predicted_class":"normal|suspected_opacity|uncertain","confidence":0.0,"visual_evidence":["obs"],"justification":"2-4 cautious sentences","limitations":["lim"],"warning":"Educational prototype only. Not for diagnosis."}
Rules:
- No patient history invention.
- No diagnosis language.
- Always fill "confidence" with a number between 0 and 1 that represents your certainty about "predicted_class" (e.g. 0.85 means 85% certain). Never leave it at 0 and never omit it.
- If confidence < 0.60, predicted_class must be "uncertain".
- Return ONLY the JSON object, nothing else."""


In [4]:
def predict_chest_xray(image_path, prompt):
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, repetition_penalty=1.3)
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return parse_prediction(text)


def parse_prediction(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)   # recupere le bloc JSON
    if not match:
        return {"predicted_class": "uncertain", "confidence": 0.0, "raw": text}
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"predicted_class": "uncertain", "confidence": 0.0, "raw": text}
    raw_pred = str(data.get("predicted_class", "")).lower()
    if "opac" in raw_pred or "abnormal" in raw_pred:
        pred = "suspected_opacity"
    elif "normal" in raw_pred:
        pred = "normal"
    else:
        pred = "uncertain"
    try:
        conf = float(data.get("confidence", 0.0))
    except (TypeError, ValueError):
        conf = 0.0
    if conf < 0.60:            # garde-fou : confiance faible => uncertain
        pred = "uncertain"
    return {"predicted_class": pred, "confidence": conf, "raw": text}


In [5]:
# A ADAPTER : dossier Kaggle qui contient train/ et train.csv
# (regarde le chemin exact dans le panneau de droite "Input" sur Kaggle)
from pathlib import Path
DATA_ROOT = next(p.parent for p in Path("/kaggle/input").rglob("train.csv"))
CSV_PATH  = DATA_ROOT / "train.csv"
print("DATA_ROOT =", DATA_ROOT)
OPACITY_COLS = ["Lung Opacity", "Consolidation", "Pneumonia", "Edema", "Atelectasis"]


def ground_truth(row):
    if row["No Finding"] == 1.0:
        return "normal"
    if any(row.get(c) == 1.0 for c in OPACITY_COLS):
        return "suspected_opacity"
    return None   # cas ni clairement sain ni opacite : exclu de l eval


def image_path_from_row(row):
    rel = row["Path"].split("/", 1)[1]   # retire le prefixe "CheXpert-v1.0-small/"
    return DATA_ROOT / rel


def build_sample(n, seed=42):
    df = pd.read_csv(CSV_PATH)
    df = df[df["Frontal/Lateral"] == "Frontal"].copy()
    df["gt"] = df.apply(ground_truth, axis=1)
    df = df[df["gt"].notna()]
    return df.sample(n=n, random_state=seed)   # meme echantillon avec le meme seed


def evaluate(prompt, sample):
    correct = committed = uncertain = 0
    for _, row in sample.iterrows():
        pred = predict_chest_xray(image_path_from_row(row), prompt)["predicted_class"]
        if pred == "uncertain":
            uncertain += 1
            continue
        committed += 1
        if pred == row["gt"]:
            correct += 1
    return {
        "accuracy": round(correct / committed, 4) if committed else 0.0,
        "coverage": round(committed / len(sample), 4),      # % de cas ou le modele s engage
        "uncertain_rate": round(uncertain / len(sample), 4),
        "n": len(sample),
    }


DATA_ROOT = /kaggle/input/datasets/ashery/chexpert


In [6]:
# Test rapide sur une image pour verifier que tout repond bien
_sample = build_sample(1, seed=0)
_row = _sample.iloc[0]
print("verite terrain :", _row["gt"])
predict_chest_xray(image_path_from_row(_row), PROMPT_BASELINE)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


verite terrain : suspected_opacity


{'predicted_class': 'suspected_opacity',
 'confidence': 1.0,
 'raw': '```json\n{\n  "image_quality": "good",\n  "predicted_class": "suspcted_opacity",\n  "confidence": 1,\n  "visual_evidence": [\n    "There appears to be increased density in the left lower lobe of the lungs compared to the right side.",\n    "The heart size seems within normal limits based on its position relative to other structures."\n  ],\n  "justification": "Based on visual inspection, there\'s some haziness present at the base of the left lung which could indicate consolidation from pneumonia but further evaluation would be needed.",\n  "limitations": [],\n  "warning": "Educational prototype only. Not for diagnosis."\n}\n```'}

In [7]:
# Evaluation baseline sur 60 images
sample = build_sample(12, seed=42)
res_baseline = evaluate(PROMPT_BASELINE, sample)
res_baseline


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


{'accuracy': 0.6667, 'coverage': 1.0, 'uncertain_rate': 0.0, 'n': 12}

In [ ]:
# Comparaison baseline vs improved sur les MEMES images
sample = build_sample(12, seed=42)
res_baseline = evaluate(PROMPT_BASELINE, sample)
res_improved = evaluate(PROMPT_IMPROVED, sample)

import pandas as pd
pd.DataFrame([
    {"prompt": "baseline", **res_baseline},
    {"prompt": "improved", **res_improved},
])